In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
 

In [2]:
def scrape_page(url, category_type):
    """Scrape a single page and extract book data"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        books = []
        
        # Each book title is inside <h3>
        items = soup.find_all('h3')
        
        for item in items:
            title = item.text.strip()
            
            # Move to parent element to get all text
            parent = item.find_parent()
            text_block = parent.get_text(separator=" ").strip()
            
            # Publisher (before "|")
            pub_match = re.search(r'\n(.*?)\s*\|', text_block)
            publisher = pub_match.group(1).strip() if pub_match else None
            
            # Published
            published_match = re.search(r'Published:\s*(.*?)(?=\n|Downloads)', text_block)
            published = published_match.group(1).strip() if published_match else None
            
            # Downloads
            downloads_match = re.search(r'Downloads:\s*(\d+)', text_block)
            downloads = downloads_match.group(1) if downloads_match else None
            
            # Pages
            pages_match = re.search(r'Pages:\s*([\d,]+)', text_block)
            pages = pages_match.group(1).strip() if pages_match else None
            
            books.append({
                "Title": title,
                "Publisher": publisher,
                "Published": published,
                "Downloads": downloads,
                "Pages": pages,
                "Category": category_type
            })
        
        return books
    
    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return []
 

In [3]:
def scrape_category(base_url, category_name, total_pages):
    """Scrape all pages for a specific category"""
    print(f"\n{'='*60}")
    print(f"Starting to scrape: {category_name} ({total_pages} pages)")
    print(f"{'='*60}")
    
    category_data = []
    
    for page in range(1, total_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}/{page}"
        
        print(f"  Scraping {category_name} - Page {page}/{total_pages}...", end=" ")
        data = scrape_page(url, category_name)
        print(f"Found {len(data)} records")
        
        category_data.extend(data)
        time.sleep(2)  # Be respectful to the server
    
    return category_data
 
 

In [4]:
# Define all categories with their page counts
categories = {
    "https://www.free-ebooks.net/philosophy": ("Philosophy", 16),
    "https://www.free-ebooks.net/history": ("History", 19),
    "https://www.free-ebooks.net/drama": ("Drama", 23),
    "https://www.free-ebooks.net/general-non-fiction": ("General Non-Fiction", 22),
    "https://www.free-ebooks.net/youth": ("Youth", 22),
    "https://www.free-ebooks.net/romance": ("Romance", 34),
    "https://www.free-ebooks.net/health": ("Health", 39),
    "https://www.free-ebooks.net/fiction": ("Fiction", 115),
    "https://www.free-ebooks.net/artificial-intelligence": ("Artificial Intelligence", 2),
    "https://www.free-ebooks.net/religious": ("Religious", 84),
    "https://www.free-ebooks.net/business": ("Business", 40),
    "https://www.free-ebooks.net/web-design": ("Web Design", 4),
    "https://www.free-ebooks.net/lgbt-studies": ("LGBT Studies", 1),
    "https://www.free-ebooks.net/crypto-blockchain": ("Crypto & Blockchain", 2),
    "https://www.free-ebooks.net/self-improvement": ("Self-Improvement", 61),
    "https://www.free-ebooks.net/career": ("Career", 9),
    "https://www.free-ebooks.net/Economy": ("Economy", 11),
    "https://www.free-ebooks.net/general-non-fiction": ("General Non Fiction", 22),
    "https://www.free-ebooks.net/communications-academic": ("Communications", 3),
    "https://www.free-ebooks.net/sociology": ("Sociology", 7),


}
 
# Scrape all categories
all_data = []
 
for base_url, (category_name, total_pages) in categories.items():
    category_data = scrape_category(base_url, category_name, total_pages)
    all_data.extend(category_data)
    print(f"✓ Completed {category_name}: {len(category_data)} total records\n")
 
# Create DataFrame and save to CSV
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total records scraped: {len(all_data)}")
 
df = pd.DataFrame(all_data)
df.to_csv("FreeMetadata_Ebooks.csv", index=False)
print(" Saved successfully to 'FreeMetadata_Ebooks.csv.csv'")
 
# Display first few records
print(f"\nFirst 5 records:")
print(df.head())
 
# Display statistics
print(f"\nRecords by Category:")
print(df['Category'].value_counts().sort_index())
 


Starting to scrape: Philosophy (16 pages)
  Scraping Philosophy - Page 1/16... Found 11 records
  Scraping Philosophy - Page 2/16... Found 11 records
  Scraping Philosophy - Page 3/16... Found 11 records
  Scraping Philosophy - Page 4/16... Found 11 records
  Scraping Philosophy - Page 5/16... Found 11 records
  Scraping Philosophy - Page 6/16... Found 11 records
  Scraping Philosophy - Page 7/16... Found 11 records
  Scraping Philosophy - Page 8/16... Found 11 records
  Scraping Philosophy - Page 9/16... Found 11 records
  Scraping Philosophy - Page 10/16... Found 11 records
  Scraping Philosophy - Page 11/16... Found 11 records
  Scraping Philosophy - Page 12/16... Found 11 records
  Scraping Philosophy - Page 13/16... Found 11 records
  Scraping Philosophy - Page 14/16... Found 11 records
  Scraping Philosophy - Page 15/16... Found 11 records
  Scraping Philosophy - Page 16/16... Found 6 records
✓ Completed Philosophy: 171 total records


Starting to scrape: History (19 pages)
  Sc